In [ ]:
import wandb
import numpy as np
import pandas as pd
import json
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# --- SELECT MODEL SIZE ---
import os

MODEL_SIZE = '7B'  # Change to '1B' or '7B'

NOTEBOOKS_DIR = (Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks')
CACHE_PATH = NOTEBOOKS_DIR / f".wandb_performance_cache_{MODEL_SIZE}.json"
USE_CACHE = False # Set to False to force re-fetch from wandb

PROJECT = "siva-reddy-mila-org/unlearning-bench"

runs_by_size = {
    '1B': {
        'Train_95frozen_FullSubset_allenai/OLMo-2-0425-1B': 'Masked Training',
        'Train_UnMask_FullSubset_allenai/OLMo-2-0425-1B': 'Unmasked Training',
    },
    '7B': {
        '7B_Train_95frozen_FullSubset_allenai/Olmo-3-1025-7B': 'Masked Training',
        '7B_UnMask_FullSubset_allenai/Olmo-3-1025-7B': 'Unmasked Training',
    },
}
selected_runs = runs_by_size[MODEL_SIZE]

interesting = [
    'hellaswag/acc,none',
    'mmlu/acc,none',
    'arc_challenge/acc,none',
    'arc_easy/acc,none',
]

if USE_CACHE and CACHE_PATH.exists():
    results_df = pd.read_json(CACHE_PATH)
    print(f"Loaded {len(results_df)} rows from cache ({CACHE_PATH})")
else:
    api = wandb.Api()
    all_runs = api.runs(PROJECT, per_page=200)

    # Filter by name, keep newest per name
    filtered = {}
    for r in all_runs:
        if r.name in selected_runs:
            if r.name not in filtered or r.created_at > filtered[r.name].created_at:
                filtered[r.name] = r

    all_results = []
    run_names = []
    pretrain = None
    for name, run in filtered.items():
        df = run.history()[interesting].dropna()
        df.columns = [col.split('/')[0] for col in df.columns]
        if pretrain is None:
            pretrain = df.iloc[0]
        all_results.append(df.iloc[-1])
        run_names.append(selected_runs[name])

    results_df = pd.DataFrame(all_results, index=run_names)
    results_df.loc['Pretrained'] = pretrain
    results_df = results_df.loc[['Pretrained'] + [idx for idx in results_df.index if idx != 'Pretrained']]

    # Save cache
    results_df.to_json(CACHE_PATH)
    print(f"Fetched {len(results_df)} rows from wandb and cached to {CACHE_PATH}")

results_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from matplotlib import rcParams
import textwrap

ASSETS_DIR = os.path.join((os.getcwd() if os.path.basename(os.getcwd()) == 'notebooks' else os.path.join(os.getcwd(), 'notebooks')), 'assets', 'imgs')
os.makedirs(ASSETS_DIR, exist_ok=True)

# 1. Global Aesthetic Settings
rcParams['font.family'] = 'monospace'
rcParams['font.monospace'] = ['Inconsolata', 'Consolas', 'DejaVu Sans Mono']
rcParams['font.style'] = 'normal'
rcParams['font.weight'] = 'bold'

# 2. DRAMATICALLY INCREASED Font Sizes (Matching previous plot)
TITLE_SIZE = 56
LABEL_SIZE = 48
TICK_SIZE = 40
BAR_LABEL_SIZE = 36
LEGEND_SIZE = 44

# 3. Data Prep
display_names = {
    'hellaswag': 'HellaSwag',
    'mmlu': 'MMLU',
    'arc_challenge': 'ARC-Challenge',
    'arc_easy': 'ARC-Easy',
}
plot_df = results_df.rename(columns=display_names)

n_runs = len(plot_df)
n_groups = len(plot_df.columns)
x = np.arange(n_groups)

# Spacing 
width = 0.8 / n_runs 

target_purple = '#662E7D'
colors = ["#BCA9F8", "#5964FF", target_purple]
TRUE_BLACK = '#000000'

# 4. High-Resolution & Large Canvas Settings (Matched to 20x12)
fig, ax = plt.subplots(figsize=(20, 12))

# 5. Plotting with precise centering
for i, (run_name, row) in enumerate(plot_df.iterrows()):
    offset = (i - (n_runs - 1) / 2) * width
    ax.bar(
        x + offset,
        row.values*100,
        width,
        label=run_name,
        color=colors[i],
        edgecolor='black',
        linewidth=2.0 # Thicker lines for large canvas
    )

# 6. Styling Adjustments
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(2)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_color(TRUE_BLACK)
ax.spines['bottom'].set_color(TRUE_BLACK)

ax.set_ylabel('Accuracy %', fontsize=LABEL_SIZE, fontweight='bold', labelpad=40)

# --- X-Axis Ticks (Text Wrapping applied here) ---
ax.set_xticks(x)

# Wrap text at ~10-12 characters to split long dataset names (like ARC-Challenge) cleanly
wrapped_labels = [textwrap.fill(str(label), width=12) for label in plot_df.columns]

ax.set_xticklabels(wrapped_labels, 
                   fontsize=TICK_SIZE, 
                   fontweight='bold', 
                   ha='center',
                   rotation=0)

# Bold, giant Y-ticks and X-ticks
ax.tick_params(axis='y', labelsize=TICK_SIZE, width=2, length=12)
ax.tick_params(axis='x', width=2, length=12, pad=15) 

ax.set_ylim(0, 100) 
ax.set_yticks(range(0, 100, 20))
ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=2)

plt.tight_layout()

# Save with model size in filename
save_path = os.path.join(ASSETS_DIR, f'performance_comparison_{MODEL_SIZE}.pdf')
fig.savefig(save_path, format='pdf', bbox_inches='tight')
print(f"Saved to {save_path}")
plt.show()

In [ ]:
plot_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import textwrap
from matplotlib import rcParams
from pathlib import Path

ASSETS_DIR = os.path.join((os.getcwd() if os.path.basename(os.getcwd()) == 'notebooks' else os.path.join(os.getcwd(), 'notebooks')), 'assets', 'imgs')
os.makedirs(ASSETS_DIR, exist_ok=True)

rcParams['font.family'] = 'monospace'
rcParams['font.monospace'] = ['Inconsolata', 'Consolas', 'DejaVu Sans Mono']
rcParams['font.style'] = 'normal'
rcParams['font.weight'] = 'bold'

LABEL_SIZE = 72
TICK_SIZE = 60
TITLE_SIZE = 76
LEGEND_SIZE = 52
TRUE_BLACK = '#000000'

colors_perf = ["#BCA9F8", "#5964FF", "#662E7D"]

display_names = {
    'hellaswag': 'HSwag', 'mmlu': 'MMLU',
    'arc_challenge': 'ARC-C', 'arc_easy': 'ARC-E',
}
col_order = ['ARC-C', 'ARC-E', 'HSwag', 'MMLU']

perf = {}
for size in ['1B', '7B']:
    df = pd.read_json(NOTEBOOKS_DIR / f".wandb_performance_cache_{size}.json")
    df = df.rename(columns=display_names)
    order = ['Pretrained'] + [i for i in df.index if i != 'Pretrained']
    perf[size] = df.loc[order, col_order]

# 0.65/39.4 = 0.33/20 = 0.0165 (equal scale factor)
fig, (ax_1b, ax_7b) = plt.subplots(1, 2, figsize=(39.4, 18), sharey=True)

def plot_perf(ax, df, title, show_ylabel=True):
    n_runs = len(df)
    n_groups = len(df.columns)
    x = np.arange(n_groups)
    width = 0.22

    for i, (run_name, row) in enumerate(df.iterrows()):
        offset = (i - (n_runs - 1) / 2) * width
        ax.bar(x + offset, row.values * 100, width,
               label=run_name, color=colors_perf[i],
               edgecolor='black', linewidth=2.5)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(3)
    ax.spines['bottom'].set_linewidth(3)
    ax.spines['left'].set_color(TRUE_BLACK)
    ax.spines['bottom'].set_color(TRUE_BLACK)

    ax.set_title(title, fontsize=TITLE_SIZE, fontweight='bold', pad=30)
    if show_ylabel:
        ax.set_ylabel('Accuracy %', fontsize=LABEL_SIZE, fontweight='bold', labelpad=40)
    else:
        plt.setp(ax.get_yticklabels(), visible=False)

    ax.set_xticks(x)
    wrapped = [textwrap.fill(str(c), width=12) for c in df.columns]
    ax.set_xticklabels(wrapped, fontsize=TICK_SIZE, fontweight='bold', ha='center')
    ax.tick_params(axis='y', labelsize=TICK_SIZE, width=3, length=14, colors=TRUE_BLACK)
    ax.tick_params(axis='x', width=3, length=14, pad=20)
    ax.set_ylim(0, 100)
    ax.set_yticks(range(0, 101, 20))
    ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=2.5)
    ax.grid(axis='x', visible=False)

plot_perf(ax_1b, perf['1B'], 'OLMo2 1B', show_ylabel=True)
plot_perf(ax_7b, perf['7B'], 'OLMo3 7B', show_ylabel=False)

handles_p, labels_p = ax_7b.get_legend_handles_labels()
ax_1b.legend(handles_p, labels_p, fontsize=LEGEND_SIZE, frameon=False,
             loc='upper left', ncol=1)

# left=0.09 → 3.5in for ylabel+ticks; bottom=0.14 shared with memo
fig.subplots_adjust(left=0.09, right=0.97, bottom=0.14, top=0.92, wspace=0.05)

out_path = os.path.join(ASSETS_DIR, 'performance_comparison_1B_7B.pdf')
fig.savefig(out_path, format='pdf')
print(f"Saved to {out_path}")
plt.show()